In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#IMPORTANT NOTE
#DATA FILES WERE ZIPPED TO SAVE DRIVE SPACE.
#FILES MUST BE DOWNLOADED, UNZIPPED, AND ACCESSED LOCALLY

# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

#read in csv of columns, dtypes - columns were manually trimmed
#df_col=pd.read_csv(r'/content/drive/My Drive/Team-Project/columns.csv')
#df_col.set_index('col',inplace=True)

#create a dictionary of {column name: dtype}
#type_dict=df_col.to_dict()['d_types']

#read in csv of data set
#df=pd.read_csv(r'/content/drive/My Drive/Team-Project/dev_data.csv',usecols=list(df_col.index.values),dtype=type_dict,na_values=na_set,)

#non-standard NA values in data set
na_set=['NULL','PrivacySuppressed']

#select the target variable
target_y='CTOTALT'

#create empty dataframe with the appropriate columns
df=pd.DataFrame(columns=pd.read_csv(r'/content/drive/My Drive/Team-Project/composite data/comp06.csv',nrows=1).columns.tolist()+['year'])

#read in csv of data set
yrs=[r'06',r'07',r'08',r'09',r'11',r'12',r'13',r'14',r'15',r'16',r'17']
for i in yrs:
  df_t=pd.read_csv(r'/content/drive/My Drive/Team-Project/composite data/comp'+i+r'.csv',na_values=na_set)
  df_t['year']=int(i)
  df=df.append(df_t,sort=False)


#drop UNITID-CIP becausethe column got messed up in data preparation
df.drop(columns='UNITID-CIP')
#get rid of any row with NaN as the target variable
df.dropna(subset=[target_y],inplace=True)

#get rid of any rows with > 25% NaN
df.dropna(thresh=36,inplace=True)

#create new index UNITID-CIP-YEAR
df['UNITID-CIP-YEAR']=df['UNITID'].astype(str)+'-'+df['CIPCODE'].astype(str)+'-'+df['year'].astype(str)
df.set_index('UNITID-CIP-YEAR',inplace=True)

#split X and y
X_w_nan=df.drop(target_y,axis=1)
y=df[target_y]

print(y.head())
# This is added for logistic regression to avoid ValueError (unknown label type)
y=y.astype('int')

print("\n")
print(y.head())


UNITID-CIP-YEAR
104531.0-11.0501-6    84
104531.0-11.0901-6    52
104531.0-11.0901-6     0
104531.0-11.0901-6     0
104531.0-15.0303-6    55
Name: CTOTALT, dtype: object


UNITID-CIP-YEAR
104531.0-11.0501-6    84
104531.0-11.0901-6    52
104531.0-11.0901-6     0
104531.0-11.0901-6     0
104531.0-15.0303-6    55
Name: CTOTALT, dtype: int64


In [ ]:
#import multivariate imputer from sklearn
#WiP: it's experimental, no promises it works well

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

#impute missing values of X
imp=IterativeImputer().fit(X_w_nan)
X=imp.transform(X_w_nan)




/usr/local/lib/python3.6/dist-packages/sklearn/impute/_iterative.py:603: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  " reached.", ConvergenceWarning)


In [ ]:
#scale data
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [ ]:
from sklearn.decomposition import IncrementalPCA
transformer = IncrementalPCA(n_components=4, batch_size=200)
#let the fit function itself divide the data into batches
X_transformed = transformer.fit_transform(X,y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size = 0.25, random_state = 0)


#Linear Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=0, solver = 'lbfgs', multi_class = 'auto', max_iter = 1500)

lr.fit(X_train, y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1500,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=0, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [ ]:
y_pred=lr.predict(X_test)

from sklearn.metrics import explained_variance_score

print(explained_variance_score(y_test,y_pred))

0.011395804691070044
